# Link Prediction on Text-Attributed Graphs — full Kaggle pipeline

Runs the entire thesis pipeline end-to-end on Kaggle: feature extraction → dataset analysis →
training all six models → test-set inference → Kaggle submission.

This notebook is a **thin driver**. It clones
[`georgegkonis/link-prediction`](https://github.com/georgegkonis/link-prediction), rewires the
repo's CWD-relative data directories onto Kaggle paths, and then shells out to the existing
`python -m scripts.…` entry points. No pipeline logic is reimplemented here, so the notebook
cannot drift from `src/`.

## Before running

| Setting | Value |
|---|---|
| **Accelerator** | GPU (T4 x2 or P100) — used by the sentence-transformer encoder |
| **Internet** | **On** — needed for `git clone`, `pip install`, NLTK and HF model downloads |
| **Input** | Add the competition **DSAA 2023 Competition** (`dsaa-2023-competition`) |
| **Secret** | Add-ons → Secrets → `GITHUB_PAT`, a GitHub token with `repo` read scope |

## Runtime

A full-scale run is measured in **hours** — the long poles are POS tagging over ~660K Wikipedia
articles, the 200-tree Random Forest, and the RBF SVM. Kaggle caps sessions at 9h (12h on GPU).

**Do a smoke test first:** set `SAMPLE_ROWS = 20_000` in the next cell and *Run All*. That
exercises every stage in minutes. Only once it is green should you clear `SAMPLE_ROWS` and
commit to a full run.

## 1. Parameters

Everything configurable lives in this cell.

In [ ]:
# ---- scale -------------------------------------------------------------------
# None  → full dataset (948,232 train / 238,364 test pairs)
# int   → truncate train.csv and test.csv to this many rows (smoke test)
SAMPLE_ROWS = None

# ---- pipeline options --------------------------------------------------------
# Node2Vec is OFF by default. It is transductive, so its embeddings are useless for test-set
# nodes unseen during training — it dropped CascadeLP's Kaggle public score from 0.99827 to
# 0.38879 — and it costs ~6.7M random walks over a 668K-node graph (slow, and the single
# biggest RAM risk in a GPU session).
#
# Caveat: the `structural` baseline in the thesis DOES use the 64-dim Node2Vec Hadamard
# features (val macro F1 0.99576). With RUN_NODE2VEC = False it is heuristics-only and scores
# ~0.769 instead. CascadeLP is unaffected either way — its thesis checkpoint is heuristics-only
# by design, so it stays that way even when Node2Vec is computed.
RUN_NODE2VEC = False

MODELS = ['structural', 'tfidf', 'pos', 'embedding', 'svm', 'cascade']

RUN_ABLATIONS    = False   # threshold grid, node2vec coverage, hard residual, throughput bench
RUN_PAPER_ASSETS = False   # regenerate latex/shared/generated_macros.tex and the thesis figures

# ---- source ------------------------------------------------------------------
REPO_OWNER  = 'georgegkonis'
REPO_NAME   = 'link-prediction'
BRANCH      = 'master'
SECRET_NAME = 'GITHUB_PAT'

# ---- where artifacts go ------------------------------------------------------
# Heavy intermediates go to /kaggle/temp, which is NOT counted against the 20GB committed
# output (pos_train.npy alone is ~546MB; pos.joblib and cascade.joblib are ~211MB each).
# The trade-off: /kaggle/temp does not survive the session, so a fresh session recomputes
# features. Point INTERIM_DIR at /kaggle/working/interim (~1.5GB) if you would rather commit
# the features and attach this notebook's output as a dataset next time.
REPO_DIR        = '/kaggle/working/' + REPO_NAME
INTERIM_DIR     = '/kaggle/temp/interim'
CHECKPOINTS_DIR = '/kaggle/temp/checkpoints'

# Derived, per model. Train and evaluate MUST agree on this value — a mismatch is a 4-vs-68
# feature-dimension crash inside StructuralClassifier — so it is computed once here and reused
# by both loops below. `cascade` is pinned to heuristics-only to match the thesis checkpoint.
NO_N2V = {m: not RUN_NODE2VEC for m in MODELS}
NO_N2V['cascade'] = True

def n2v_flag(model: str) -> str:
    return f'training.no_n2v={str(NO_N2V[model]).lower()}'

print(f'scale       : {"FULL" if SAMPLE_ROWS is None else f"SAMPLE ({SAMPLE_ROWS:,} rows)"}')
print(f'models      : {", ".join(MODELS)}')
print(f'node2vec    : {"computed" if RUN_NODE2VEC else "skipped"}')
print(f'no_n2v      : ' + ', '.join(f'{m}={str(NO_N2V[m]).lower()}' for m in MODELS))

## 2. Dependencies

Kaggle's image has PyTorch, scikit-learn, NLTK and sentence-transformers, but not `hydra-core`
(the config system behind `scripts/train.py` and `scripts/evaluate.py`) or `gensim`. `gensim` is
imported at the top of `src/features/structural.py`, so it is required even when Node2Vec is
skipped.

In [ ]:
import importlib.util
import subprocess
import sys

def _missing(*mods):
    return [m for m in mods if importlib.util.find_spec(m.replace('-', '_')) is None]

need = _missing('hydra', 'gensim', 'sentence_transformers')
need = ['hydra-core' if m == 'hydra' else m for m in need]

if need:
    print('installing:', need)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
else:
    print('all dependencies already present')

## 3. Clone the repository

The PAT is read from Kaggle Secrets and embedded in the clone URL. It is never printed — output
is scrubbed of the token before anything is displayed.

In [ ]:
import os
import shutil
from pathlib import Path

from kaggle_secrets import UserSecretsClient

PAT = UserSecretsClient().get_secret(SECRET_NAME)
assert PAT, f'secret {SECRET_NAME!r} is empty — add it under Add-ons → Secrets'

def _scrub(text: str) -> str:
    return text.replace(PAT, '***')

if Path(REPO_DIR, '.git').exists():
    print(f'{REPO_DIR} already cloned — pulling latest')
    cmd = ['git', '-C', REPO_DIR, 'pull', '--quiet', '--ff-only']
else:
    url = f'https://{PAT}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    cmd = ['git', 'clone', '--quiet', '--depth', '1', '--branch', BRANCH, url, REPO_DIR]

proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(_scrub(proc.stderr or proc.stdout))

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

head = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout
print('cwd :', os.getcwd())
print('HEAD:', head.strip())

## 4. Wire Kaggle paths into the repo layout

The pipeline has no central paths module — `'data/raw'`, `'data/interim'` and `'outputs/…'` are
duplicated as CWD-relative constants across the scripts (and mirrored in `configs/config.yaml`).
Rather than patch every one, we make those directories point at the right places:

- `data/raw/` — symlinks to the read-only competition input. Note the competition ships
  `nodes.tsv` inside a `nodes/` subdirectory, so it is linked file-by-file rather than as a
  directory.
- `data/interim/` → `INTERIM_DIR`, `outputs/checkpoints/` → `CHECKPOINTS_DIR`.
- `outputs/predictions/` stays inside `/kaggle/working` so metrics and submissions are committed.

In sample mode `data/raw/train.csv` and `test.csv` become truncated **copies**. Because the
feature scripts and Hydra both read the same `data/raw`, features and training can never
disagree about which rows exist.

In [ ]:
import pandas as pd

# --- locate the competition input -------------------------------------------------
# Searched recursively: the mount point depends on how the data was attached — a competition
# added to the notebook lands at /kaggle/input/competitions/<slug>/, a dataset at
# /kaggle/input/<slug>/.
candidates = sorted(Path('/kaggle/input').rglob('train.csv'))
assert candidates, 'no train.csv under /kaggle/input — attach the dsaa-2023-competition data'
INPUT_DIR = candidates[0].parent

nodes_src = next(iter(sorted(INPUT_DIR.rglob('nodes.tsv'))), None)
assert nodes_src, f'nodes.tsv not found under {INPUT_DIR}'
print('input:', INPUT_DIR)

# --- link the writable directories -------------------------------------------------
def link_dir(inside_repo: str, target: str):
    """Replace <repo>/inside_repo with a symlink to target (created if absent)."""
    Path(target).mkdir(parents=True, exist_ok=True)
    p = Path(inside_repo)
    if p.is_symlink():
        p.unlink()
    elif p.exists():
        shutil.rmtree(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.symlink_to(target, target_is_directory=True)

link_dir('data/interim', INTERIM_DIR)
link_dir('outputs/checkpoints', CHECKPOINTS_DIR)
Path('outputs/predictions').mkdir(parents=True, exist_ok=True)
Path('logs').mkdir(exist_ok=True)          # hydra's job_logging writes logs/<job>.log

# --- build data/raw ---------------------------------------------------------------
RAW = Path('data/raw')
if RAW.is_symlink():
    RAW.unlink()
elif RAW.exists():
    shutil.rmtree(RAW)
RAW.mkdir(parents=True)

(RAW / 'nodes.tsv').symlink_to(nodes_src)
(RAW / 'sample_submission.csv').symlink_to(INPUT_DIR / 'sample_submission.csv')

for name in ('train.csv', 'test.csv'):
    src = INPUT_DIR / name
    if SAMPLE_ROWS is None:
        (RAW / name).symlink_to(src)
    else:
        pd.read_csv(src, nrows=SAMPLE_ROWS).to_csv(RAW / name, index=False)

for f in sorted(RAW.iterdir()):
    kind = '→ ' + str(Path(os.readlink(f)).parent) if f.is_symlink() else 'copy'
    print(f'  {f.name:24s} {f.stat().st_size / 1e6:9.1f} MB  {kind}')

### A shell helper

Every stage below is an existing entry point. `sh()` streams its output line by line so long
stages show live `tqdm` progress, and raises on a non-zero exit so a failure stops *Run All*
instead of silently poisoning later cells.

In [ ]:
import time

def sh(*args: str):
    """Run a command in the repo root, streaming output; raise on failure."""
    cmd = [sys.executable, *args] if args[0] == '-m' else list(args)
    label = args[1] if args[0] == '-m' else args[0]
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    mins = (time.time() - t0) / 60
    if proc.returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {proc.returncode} after {mins:.1f} min')
    print(f'\n[done in {mins:.1f} min]', flush=True)

## 5. Sanity check

Confirms the environment before anything expensive starts.

In [ ]:
import multiprocessing

train = pd.read_csv('data/raw/train.csv')
test  = pd.read_csv('data/raw/test.csv')
print(f'train pairs : {len(train):,}   positives: {train["label"].mean():.1%}')
print(f'test pairs  : {len(test):,}')
print(f'self-loops  : train {(train.id1 == train.id2).sum():,}  test {(test.id1 == test.id2).sum():,}')
print(f'cpus        : {multiprocessing.cpu_count()}')
print('ram         :', subprocess.run(['free', '-h'], capture_output=True, text=True).stdout.splitlines()[1])
print('disk        :', subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True).stdout.splitlines()[1])

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
print('gpu         :', gpu.stdout.strip() or 'NONE — enable a GPU accelerator, or expect a very slow encoder')

# Pre-warm the NLTK models used by the POS features (needs internet).
subprocess.run(
    [sys.executable, '-c', 'from src.features.linguistic import ensure_nltk_data; ensure_nltk_data()'],
    cwd=REPO_DIR, check=True,
)
print('nltk        : punkt_tab + averaged_perceptron_tagger_eng ready')
del train, test

## 6. Structural features

`compute_heuristics` builds the positive-edge graph and computes Common Neighbours, Jaccard,
Adamic-Adar and Preferential Attachment for every train and test pair
→ `data/interim/structural_{train,test}.csv`.

With `RUN_NODE2VEC = True` this also trains the DeepWalk-style embedding
(`node2vec.kv`) and writes the 64-dim Hadamard features `n2v_{train,test}.npy`.

In [ ]:
args = ['-m', 'scripts.data.compute_structural']
if not RUN_NODE2VEC:
    args.append('--skip-n2v')
sh(*args)

## 7. Semantic and linguistic features

One pass over the 671MB `nodes.tsv` (streamed in 50K-row chunks, keeping only the ~660K node IDs
that actually appear in a pair), then three feature families:

| Output | Feature |
|---|---|
| `tfidf_{train,test}.csv` | TF-IDF cosine similarity (50K vocab, `sublinear_tf`, `min_df=2`) |
| `sentence_emb_{train,test}.csv` | `all-MiniLM-L6-v2` cosine similarity (GPU) |
| `pos_{train,test}.npy` | 72-dim POS frequency vector (36 Penn tags per node, concatenated) |

**This is the slowest cell in the notebook.** POS tagging is cached per unique node, but it still
means running the NLTK tokenizer and perceptron tagger over every article.

In [ ]:
sh('-m', 'scripts.data.compute_semantic')

## 8. Dataset analysis

- `audit_leakage` — train/test pair overlap, self-loop counts, intra-train duplicates.
- `analyze_dataset` — data-driven trivial/hard thresholds at 1% FPR on the negative class,
  writing `difficulty_{train,test}.csv` and `difficulty_thresholds.json`.

In [ ]:
sh('-m', 'scripts.analysis.audit_leakage')
sh('-m', 'scripts.analysis.analyze_dataset')

## 9. Train

Each model is trained on a stratified 80/20 split with self-loops held out (they are handled
separately by CascadeLP's Tier 0). Per model this writes
`outputs/checkpoints/<model>.joblib` and `outputs/predictions/<model>_val_metrics.json`;
`cascade` additionally exports per-pair tier/difficulty/correctness to `cascade_val_tiers.csv`,
and `svm` exports `svm_val_errors.csv`.

Expect `svm` to be slow — the RBF kernel is fitted on a stratified 20K subsample precisely
because SVC is O(n²–n³), but `predict_proba` over 187K validation pairs still dominates its
latency.

In [ ]:
for model in MODELS:
    print(f'\n{"=" * 78}\n TRAIN {model}\n{"=" * 78}')
    sh('-m', 'scripts.train', f'model={model}', n2v_flag(model))

## 10. Evaluate — test-set inference and submissions

Loads each checkpoint, runs inference over the test pairs and writes
`outputs/predictions/<model>_submission.csv`. `cascade` also exports
`cascade_test_tiers.csv` (per-pair tier, difficulty and confidence).

In [ ]:
for model in MODELS:
    print(f'\n{"=" * 78}\n EVALUATE {model}\n{"=" * 78}')
    sh('-m', 'scripts.evaluate', f'model={model}', n2v_flag(model))

## 11. CascadeLP analysis

`analyze_cascade` reads `cascade_val_tiers.csv` and reports which tier resolves which pairs,
cross-referenced against the measured difficulty label — the cost-aware diagnostic that is the
thesis's framing of the contribution.

The ablations behind `RUN_ABLATIONS` are each a substantial extra run: a threshold grid search,
Node2Vec coverage, the hard-residual breakdown, and a throughput benchmark.

In [ ]:
if 'cascade' in MODELS:
    sh('-m', 'scripts.analysis.analyze_cascade')

if RUN_ABLATIONS and 'cascade' in MODELS:
    sh('-m', 'scripts.analysis.ablate_cascade_thresholds')
    sh('-m', 'scripts.analysis.analyze_hard_residual')
    sh('-m', 'scripts.analysis.benchmark_throughput')
    if RUN_NODE2VEC:
        sh('-m', 'scripts.analysis.ablate_node2vec')
else:
    print('ablations skipped (RUN_ABLATIONS = False)')

## 12. Results

Validation metrics for every model, plus the CascadeLP tier distribution on the test set.

Reference values from a full-scale local run (macro F1): `pos` 0.9988, `structural` 0.9958,
`svm` 0.9091, `tfidf` 0.9090, `embedding` 0.9058. CascadeLP's Kaggle public score was 0.99827.

Two things shift these numbers:

- With `RUN_NODE2VEC = False`, `structural` is heuristics-only and lands near **0.769**, not
  0.9958 — the thesis figure includes the Node2Vec features.
- A sample-mode run lands well below all of them: truncating `train.csv` leaves a far sparser
  graph, so nearly every pair becomes cold-start.

In [ ]:
import json

PRED = Path('outputs/predictions')

rows = []
for f in sorted(PRED.glob('*_val_metrics.json')):
    rows.append({'model': f.name.removesuffix('_val_metrics.json'), **json.loads(f.read_text())})

results = pd.DataFrame(rows).sort_values('macro_f1', ascending=False).set_index('model')
display(results.style.format({
    'macro_f1': '{:.5f}', 'auc_roc': '{:.5f}', 'cold_start_f1': '{:.5f}',
    'latency_ms': '{:,.1f}', 'n_val': '{:,.0f}',
}))

tiers_path = PRED / 'cascade_test_tiers.csv'
if tiers_path.exists():
    tiers = pd.read_csv(tiers_path)
    print('\nCascadeLP tier distribution on the test set:')
    dist = tiers.tier_used.value_counts().sort_index()
    for tier, n in dist.items():
        print(f'  tier {tier}: {n:>8,} pairs ({n / len(tiers):6.1%})')
    print('\nTier x difficulty:')
    display(pd.crosstab(tiers.tier_used, tiers.difficulty))

## 13. Submission

`submission.csv` is written to `/kaggle/working` so this notebook can be submitted directly from
the Kaggle UI (*Submit to Competition*) — no API token needed. CascadeLP is the submitted model;
change `SUBMIT_MODEL` to submit a baseline instead.

Every other model's submission, metrics and per-pair export is copied to
`/kaggle/working/predictions/` so it survives the session.

In [ ]:
SUBMIT_MODEL = 'cascade' if 'cascade' in MODELS else MODELS[-1]

src = PRED / f'{SUBMIT_MODEL}_submission.csv'
assert src.exists(), f'{src} missing — did evaluate run for {SUBMIT_MODEL}?'

submission = pd.read_csv(src)
n_test = len(pd.read_csv('data/raw/test.csv'))
assert list(submission.columns) == ['id', 'label'], submission.columns.tolist()
assert len(submission) == n_test, f'{len(submission):,} rows != {n_test:,} test pairs'
assert submission.label.isin([0, 1]).all(), 'labels must be 0/1'

out = Path('/kaggle/working/submission.csv')
submission.to_csv(out, index=False)
print(f'{out}  ←  {SUBMIT_MODEL}   {len(submission):,} rows, {submission.label.mean():.1%} positive')

# Persist everything else outside the /kaggle/temp scratch space.
keep = Path('/kaggle/working/predictions')
keep.mkdir(exist_ok=True)
for f in PRED.iterdir():
    if f.is_file():
        shutil.copy2(f, keep / f.name)
print(f'copied {len(list(keep.iterdir()))} files → {keep}')
display(submission.head())

## 14. Optional — thesis assets

Regenerates `latex/shared/generated_macros.tex` (every numeric claim in the thesis resolves to a macro
defined there) and the figures, from the outputs produced above. No LaTeX compile.

Note that `generate_macros.py` reads several artifacts that only a full-scale run produces; on a
sample run it will report missing inputs.

In [ ]:
if RUN_PAPER_ASSETS:
    sh('-m', 'scripts.paper.generate_macros')
    sh('-m', 'scripts.paper.generate_figures')

    from IPython.display import Image, display as _d
    for png in sorted(Path('outputs/figures').glob('*.png')):
        print(png.name)
        _d(Image(filename=str(png), width=760))
else:
    print('paper assets skipped (RUN_PAPER_ASSETS = False)')